In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

import utils.data_processing_bronze_table
import utils.data_processing_silver_table
import utils.data_processing_gold_table

## set up pyspark session

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/14 14:23:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## set up config

In [3]:
# set up config
start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [4]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
print(dates_str_lst)

['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01', '2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01', '2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']


## EDA on raw data
check each source csv before building the pipeline, cleaning rules in the silver layer come from what we find here

In [5]:
# load raw csvs to check size and coverage
lms_raw = pd.read_csv("data/lms_loan_daily.csv")
clks_raw = pd.read_csv("data/feature_clickstream.csv")
attr_raw = pd.read_csv("data/features_attributes.csv")
fin_raw = pd.read_csv("data/features_financials.csv")

print("lms:", lms_raw.shape, "| customers:", lms_raw.Customer_ID.nunique())
print("clickstream:", clks_raw.shape, "| customers:", clks_raw.Customer_ID.nunique())
print("attributes:", attr_raw.shape, "| customers:", attr_raw.Customer_ID.nunique())
print("financials:", fin_raw.shape, "| customers:", fin_raw.Customer_ID.nunique())

lms: (137500, 11) | customers: 12500
clickstream: (215376, 22) | customers: 8974
attributes: (12500, 6) | customers: 12500
financials: (12500, 22) | customers: 12500


In [6]:
# attributes issues: garbled SSN, underscores in Age, placeholder Occupation
attr_raw[attr_raw.SSN == "#F%$D@*&8"].head(3)

,Customer_ID,Name,Age,SSN,Occupation,snapshot_date
2,CUS_0x100b,Shirboni,19,#F%$D@*&8,Media_Manager,2024-03-01
9,CUS_0x102e,Rhysn,26,#F%$D@*&8,Scientist,2024-04-01
15,CUS_0x1044,Maki Shirakip,44,#F%$D@*&8,_______,2023-06-01


In [7]:
# Age is stored as string, some values have trailing underscores and some are impossible
age_num = pd.to_numeric(attr_raw.Age.astype(str).str.replace("_",""), errors="coerce")
print("Age values with underscore:", attr_raw.Age.astype(str).str.contains("_").sum())
print("Age min:", age_num.min(), "max:", age_num.max())
print("Age outside 18-100:", ((age_num < 18) | (age_num > 100)).sum())
print("garbled SSN count:", (attr_raw.SSN == "#F%$D@*&8").sum())
print(attr_raw.Occupation.value_counts().head(20))

Age values with underscore: 637
Age min: -500 max: 8678
Age outside 18-100: 988
garbled SSN count: 703
Occupation
_______          880
Lawyer           828
Architect        795
Engineer         793
Accountant       791
Scientist        789
Teacher          782
Media_Manager    780
Mechanic         780
Developer        780
Entrepreneur     776
Journalist       761
Doctor           760
Musician         741
Manager          736
Writer           728
Name: count, dtype: int64


In [8]:
# financials issues: numbers stored as strings with underscores, junk categories, impossible values
print("Annual_Income with underscore:", fin_raw.Annual_Income.astype(str).str.contains("_").sum())
print(fin_raw[fin_raw.Annual_Income.astype(str).str.contains("_")][["Customer_ID","Annual_Income"]].head(5))
print()
print("Credit_Mix values:", fin_raw.Credit_Mix.unique())
print("Payment_Behaviour values:", fin_raw.Payment_Behaviour.unique())
print()
print("Credit_History_Age sample:", fin_raw.Credit_History_Age.dropna().iloc[0])

Annual_Income with underscore: 859
   Customer_ID Annual_Income
1   CUS_0x1009     52312.68_
29  CUS_0x107c     49718.55_
34  CUS_0x1098     20652.98_
51  CUS_0x10eb     28315.95_
56  CUS_0x1100     43062.54_

Credit_Mix values: ['Bad' '_' 'Good' 'Standard']
Payment_Behaviour values: ['High_spent_Medium_value_payments' 'High_spent_Small_value_payments'
 'Low_spent_Medium_value_payments' 'Low_spent_Small_value_payments'
 '!@9#%8' 'High_spent_Large_value_payments'
 'Low_spent_Large_value_payments']

Credit_History_Age sample: 10 Years and 9 Months


In [9]:
# check ranges, the corrupted values sit far away from the real ones
inc = pd.to_numeric(fin_raw.Annual_Income.astype(str).str.replace("_",""), errors="coerce")
print("Annual_Income p99:", round(inc.quantile(0.99)), "| max:", round(inc.max()))
print("Num_Bank_Accounts min:", fin_raw.Num_Bank_Accounts.min(), "| max:", fin_raw.Num_Bank_Accounts.max())
print("Num_Credit_Card max:", fin_raw.Num_Credit_Card.max())
print("Interest_Rate max:", fin_raw.Interest_Rate.max())
ndp = pd.to_numeric(fin_raw.Num_of_Delayed_Payment.astype(str).str.replace("_",""), errors="coerce")
print("Num_of_Delayed_Payment min:", ndp.min(), "| max:", ndp.max())
print("Delay_from_due_date min:", fin_raw.Delay_from_due_date.min(), "(negative = paid early, this one is legit)")

Annual_Income p99: 178816 | max: 23834698
Num_Bank_Accounts min: -1 | max: 1756
Num_Credit_Card max: 1499
Interest_Rate max: 5789
Num_of_Delayed_Payment min: -3 | max: 4293
Delay_from_due_date min: -5 (negative = paid early, this one is legit)


In [10]:
# clickstream coverage: not every customer has clickstream history
clk_customers = set(clks_raw.Customer_ID)
all_customers = set(attr_raw.Customer_ID)
print("customers with clickstream:", len(clk_customers))
print("customers without clickstream:", len(all_customers - clk_customers))
print("coverage:", round(len(clk_customers) / len(all_customers) * 100, 1), "%")
print("snapshots per customer:", clks_raw.groupby("Customer_ID").snapshot_date.nunique().describe()[["min","max"]].to_dict())

customers with clickstream: 8974
customers without clickstream: 3526
coverage: 71.8 %
snapshots per customer: {'min': 24.0, 'max': 24.0}


## Build Bronze Table

In [11]:
# create bronze datalake
bronze_lms_directory = "datamart/bronze/lms/"

if not os.path.exists(bronze_lms_directory):
    os.makedirs(bronze_lms_directory)

bronze_clickstream_directory = "datamart/bronze/clickstream/"

if not os.path.exists(bronze_clickstream_directory):
    os.makedirs(bronze_clickstream_directory)

bronze_attributes_directory = "datamart/bronze/attributes/"

if not os.path.exists(bronze_attributes_directory):
    os.makedirs(bronze_attributes_directory)

bronze_financials_directory = "datamart/bronze/financials/"

if not os.path.exists(bronze_financials_directory):
    os.makedirs(bronze_financials_directory)

In [12]:
# run bronze backfill for all 4 sources
for date_str in dates_str_lst:
    utils.data_processing_bronze_table.process_bronze_table(date_str, bronze_lms_directory, spark)
    utils.data_processing_bronze_table.process_bronze_clickstream(date_str, bronze_clickstream_directory, spark)
    utils.data_processing_bronze_table.process_bronze_attributes(date_str, bronze_attributes_directory, spark)
    utils.data_processing_bronze_table.process_bronze_financials(date_str, bronze_financials_directory, spark)

2023-01-01row count: 530
saved to: datamart/bronze/lms/bronze_loan_daily_2023_01_01.csv
2023-01-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_2023_01_01.csv
2023-01-01row count: 530
saved to: datamart/bronze/attributes/bronze_attributes_2023_01_01.csv
2023-01-01row count: 530
saved to: datamart/bronze/financials/bronze_financials_2023_01_01.csv
2023-02-01row count: 1031
saved to: datamart/bronze/lms/bronze_loan_daily_2023_02_01.csv
2023-02-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_2023_02_01.csv
2023-02-01row count: 501
saved to: datamart/bronze/attributes/bronze_attributes_2023_02_01.csv
2023-02-01row count: 501
saved to: datamart/bronze/financials/bronze_financials_2023_02_01.csv
2023-03-01row count: 1537
saved to: datamart/bronze/lms/bronze_loan_daily_2023_03_01.csv
2023-03-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_2023_03_01.csv
2023-03-01row count: 506
saved to: datamart/bronze/attribute

2024-03-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_2024_03_01.csv
2024-03-01row count: 511
saved to: datamart/bronze/attributes/bronze_attributes_2024_03_01.csv
2024-03-01row count: 511
saved to: datamart/bronze/financials/bronze_financials_2024_03_01.csv
2024-04-01row count: 5417
saved to: datamart/bronze/lms/bronze_loan_daily_2024_04_01.csv
2024-04-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_2024_04_01.csv
2024-04-01row count: 513
saved to: datamart/bronze/attributes/bronze_attributes_2024_04_01.csv
2024-04-01row count: 513
saved to: datamart/bronze/financials/bronze_financials_2024_04_01.csv
2024-05-01row count: 5391
saved to: datamart/bronze/lms/bronze_loan_daily_2024_05_01.csv
2024-05-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_2024_05_01.csv
2024-05-01row count: 491
saved to: datamart/bronze/attributes/bronze_attributes_2024_05_01.csv
2024-05-01row count: 491
saved to: datamart/bronze/fi

2024-10-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_2024_10_01.csv
2024-10-01row count: 456
saved to: datamart/bronze/attributes/bronze_attributes_2024_10_01.csv
2024-10-01row count: 456
saved to: datamart/bronze/financials/bronze_financials_2024_10_01.csv
2024-11-01row count: 5501
saved to: datamart/bronze/lms/bronze_loan_daily_2024_11_01.csv
2024-11-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_2024_11_01.csv
2024-11-01row count: 488
saved to: datamart/bronze/attributes/bronze_attributes_2024_11_01.csv
2024-11-01row count: 488
saved to: datamart/bronze/financials/bronze_financials_2024_11_01.csv
2024-12-01row count: 5531
saved to: datamart/bronze/lms/bronze_loan_daily_2024_12_01.csv
2024-12-01row count: 8974
saved to: datamart/bronze/clickstream/bronze_clickstream_2024_12_01.csv
2024-12-01row count: 515
saved to: datamart/bronze/attributes/bronze_attributes_2024_12_01.csv
2024-12-01row count: 515
saved to: datamart/bronze/fi

In [13]:
# inspect output
df = spark.read.csv(bronze_financials_directory + "bronze_financials_2023_05_01.csv", header=True, inferSchema=True)
df.select("Customer_ID", "Annual_Income", "Num_Bank_Accounts", "Credit_Mix", "Payment_Behaviour", "Credit_History_Age").show(10)

+-----------+-------------+-----------------+----------+--------------------+--------------------+
|Customer_ID|Annual_Income|Num_Bank_Accounts|Credit_Mix|   Payment_Behaviour|  Credit_History_Age|
+-----------+-------------+-----------------+----------+--------------------+--------------------+
| CUS_0x1000|     30625.94|                6|       Bad|High_spent_Medium...|10 Years and 9 Mo...|
| CUS_0x108a|     36982.36|                7|         _|Low_spent_Medium_...|7 Years and 9 Months|
| CUS_0x10f9|    150131.68|                5|      Good|High_spent_Medium...|31 Years and 11 M...|
| CUS_0x1119|      56301.9|                9|         _|Low_spent_Medium_...|14 Years and 6 Mo...|
| CUS_0x1192|    16319.375|                7|  Standard|Low_spent_Medium_...|23 Years and 9 Mo...|
| CUS_0x11b1|     34819.83|              312|         _|High_spent_Small_...|18 Years and 10 M...|
| CUS_0x12a7|     26450.59|                5|  Standard|High_spent_Large_...|28 Years and 8 Mo...|
| CUS_0x12

## Build Silver Table

In [14]:
# create silver datalake
silver_loan_daily_directory = "datamart/silver/loan_daily/"

if not os.path.exists(silver_loan_daily_directory):
    os.makedirs(silver_loan_daily_directory)

silver_clickstream_directory = "datamart/silver/clickstream/"

if not os.path.exists(silver_clickstream_directory):
    os.makedirs(silver_clickstream_directory)

silver_attributes_directory = "datamart/silver/attributes/"

if not os.path.exists(silver_attributes_directory):
    os.makedirs(silver_attributes_directory)

silver_financials_directory = "datamart/silver/financials/"

if not os.path.exists(silver_financials_directory):
    os.makedirs(silver_financials_directory)

In [15]:
# run silver backfill, clean each source separately
for date_str in dates_str_lst:
    utils.data_processing_silver_table.process_silver_table(date_str, bronze_lms_directory, silver_loan_daily_directory, spark)
    utils.data_processing_silver_table.process_silver_clickstream(date_str, bronze_clickstream_directory, silver_clickstream_directory, spark)
    utils.data_processing_silver_table.process_silver_attributes(date_str, bronze_attributes_directory, silver_attributes_directory, spark)
    utils.data_processing_silver_table.process_silver_financials(date_str, bronze_financials_directory, silver_financials_directory, spark)

loaded from: datamart/bronze/lms/bronze_loan_daily_2023_01_01.csv row count: 530


saved to: datamart/silver/loan_daily/silver_loan_daily_2023_01_01.parquet
loaded from: datamart/bronze/clickstream/bronze_clickstream_2023_01_01.csv row count: 8974


saved to: datamart/silver/clickstream/silver_clickstream_2023_01_01.parquet
loaded from: datamart/bronze/attributes/bronze_attributes_2023_01_01.csv row count: 530


saved to: datamart/silver/attributes/silver_attributes_2023_01_01.parquet
loaded from: datamart/bronze/financials/bronze_financials_2023_01_01.csv row count: 530


saved to: datamart/silver/financials/silver_financials_2023_01_01.parquet
loaded from: datamart/bronze/lms/bronze_loan_daily_2023_02_01.csv row count: 1031


...

saved to: datamart/silver/financials/silver_financials_2024_11_01.parquet
loaded from: datamart/bronze/lms/bronze_loan_daily_2024_12_01.csv row count: 5531


saved to: datamart/silver/loan_daily/silver_loan_daily_2024_12_01.parquet
loaded from: datamart/bronze/clickstream/bronze_clickstream_2024_12_01.csv row count: 8974


saved to: datamart/silver/clickstream/silver_clickstream_2024_12_01.parquet
loaded from: datamart/bronze/attributes/bronze_attributes_2024_12_01.csv row count: 515


saved to: datamart/silver/attributes/silver_attributes_2024_12_01.parquet
loaded from: datamart/bronze/financials/bronze_financials_2024_12_01.csv row count: 515


saved to: datamart/silver/financials/silver_financials_2024_12_01.parquet


In [16]:
# inspect output: same financials columns after cleaning, underscores gone, junk categories now null
df = spark.read.parquet(silver_financials_directory + "silver_financials_2023_05_01.parquet")
df.select("Customer_ID", "Annual_Income", "Num_Bank_Accounts", "Credit_Mix", "Payment_Behaviour", "Credit_History_Months", "Num_Loan_Types").show(10)

+-----------+-------------+-----------------+----------+--------------------+---------------------+--------------+
|Customer_ID|Annual_Income|Num_Bank_Accounts|Credit_Mix|   Payment_Behaviour|Credit_History_Months|Num_Loan_Types|
+-----------+-------------+-----------------+----------+--------------------+---------------------+--------------+
| CUS_0x1000|     30625.94|                6|       Bad|High_spent_Medium...|                  129|             2|
| CUS_0x108a|     36982.36|                7|      NULL|Low_spent_Medium_...|                   93|             9|
| CUS_0x10f9|    150131.69|                5|      Good|High_spent_Medium...|                  383|             0|
| CUS_0x1119|      56301.9|                9|      NULL|Low_spent_Medium_...|                  174|             2|
| CUS_0x1192|    16319.375|                7|  Standard|Low_spent_Medium_...|                  285|             2|
| CUS_0x11b1|     34819.83|             NULL|      NULL|High_spent_Small_...|   

In [17]:
# inspect output: cleaned attributes, PII dropped
df = spark.read.parquet(silver_attributes_directory + "silver_attributes_2023_05_01.parquet")
df.show(10)
df.printSchema()

+-----------+----+-------------+-------------+
|Customer_ID| Age|   Occupation|snapshot_date|
+-----------+----+-------------+-------------+
| CUS_0x1000|  18|       Lawyer|   2023-05-01|
| CUS_0x108a|  38|   Journalist|   2023-05-01|
| CUS_0x10f9|  54|         NULL|   2023-05-01|
| CUS_0x1119|  36| Entrepreneur|   2023-05-01|
| CUS_0x1192|NULL|Media_Manager|   2023-05-01|
| CUS_0x11b1|  28|         NULL|   2023-05-01|
| CUS_0x12a7|  37|Media_Manager|   2023-05-01|
| CUS_0x12a9|  44|Media_Manager|   2023-05-01|
| CUS_0x1340|  42|    Architect|   2023-05-01|
| CUS_0x13d1|  22|     Musician|   2023-05-01|
+-----------+----+-------------+-------------+
only showing top 10 rows

root
 |-- Customer_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)



## Build gold table for labels and features

In [18]:
# create gold datalake
gold_label_store_directory = "datamart/gold/label_store/"

if not os.path.exists(gold_label_store_directory):
    os.makedirs(gold_label_store_directory)

gold_feature_store_directory = "datamart/gold/feature_store/"

if not os.path.exists(gold_feature_store_directory):
    os.makedirs(gold_feature_store_directory)

In [19]:
# run gold backfill for label store and feature store
for date_str in dates_str_lst:
    utils.data_processing_gold_table.process_labels_gold_table(date_str, silver_loan_daily_directory, gold_label_store_directory, spark, dpd = 30, mob = 6)
    utils.data_processing_gold_table.process_features_gold_table(date_str, silver_attributes_directory, silver_financials_directory, silver_clickstream_directory, gold_feature_store_directory, spark)

loaded from: datamart/silver/loan_daily/silver_loan_daily_2023_01_01.parquet row count: 530


saved to: datamart/gold/label_store/gold_label_store_2023_01_01.parquet
loaded from: datamart/silver/attributes/silver_attributes_2023_01_01.parquet row count: 530
loaded from: datamart/silver/financials/silver_financials_2023_01_01.parquet row count: 530


saved to: datamart/gold/feature_store/gold_feature_store_2023_01_01.parquet
loaded from: datamart/silver/loan_daily/silver_loan_daily_2023_02_01.parquet row count: 1031


saved to: datamart/gold/label_store/gold_label_store_2023_02_01.parquet
loaded from: datamart/silver/attributes/silver_attributes_2023_02_01.parquet row count: 501
loaded from: datamart/silver/financials/silver_financials_2023_02_01.parquet row count: 501


saved to: datamart/gold/feature_store/gold_feature_store_2023_02_01.parquet
loaded from: datamart/silver/loan_daily/silver_loan_daily_2023_03_01.parquet row count: 1537


...

saved to: datamart/gold/feature_store/gold_feature_store_2024_10_01.parquet
loaded from: datamart/silver/loan_daily/silver_loan_daily_2024_11_01.parquet row count: 5501


saved to: datamart/gold/label_store/gold_label_store_2024_11_01.parquet
loaded from: datamart/silver/attributes/silver_attributes_2024_11_01.parquet row count: 488
loaded from: datamart/silver/financials/silver_financials_2024_11_01.parquet row count: 488


saved to: datamart/gold/feature_store/gold_feature_store_2024_11_01.parquet
loaded from: datamart/silver/loan_daily/silver_loan_daily_2024_12_01.parquet row count: 5531


saved to: datamart/gold/label_store/gold_label_store_2024_12_01.parquet
loaded from: datamart/silver/attributes/silver_attributes_2024_12_01.parquet row count: 515
loaded from: datamart/silver/financials/silver_financials_2024_12_01.parquet row count: 515


saved to: datamart/gold/feature_store/gold_feature_store_2024_12_01.parquet


## inspect label store

In [20]:
folder_path = gold_label_store_directory
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("label store row_count:", df.count())
df.show()

label store row_count: 8974
+--------------------+-----------+-----+----------+-------------+
|             loan_id|Customer_ID|label| label_def|snapshot_date|
+--------------------+-----------+-----+----------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|   2023-07-01|
|CUS_0x13a8_2023_0...| CUS_0x13a8|    0|30dpd_6mob|   2023-07-01|
|CUS_0x13ef_2023_0...| CUS_0x13ef|    0|30dpd_6mob|   2023-07-01|
|CUS_0x1440_2023_0...| CUS_0x1440|    0|30dpd_6m

## inspect feature store

In [21]:
folder_path = gold_feature_store_directory
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
feat_df = spark.read.option("header", "true").parquet(*files_list)
print("feature store row_count:", feat_df.count())
feat_df.select("Customer_ID", "Age", "Occupation", "Annual_Income", "Credit_Mix",
               "debt_to_income", "emi_to_salary", "loans_per_credit_year", "num_credit_products", "has_clickstream",
               "snapshot_date").show()

feature store row_count: 11974
+-----------+---+-------------+-------------+----------+--------------+-------------+---------------------+-------------------+---------------+-------------+
|Customer_ID|Age|   Occupation|Annual_Income|Credit_Mix|debt_to_income|emi_to_salary|loans_per_credit_year|num_credit_products|has_clickstream|snapshot_date|
+-----------+---+-------------+-------------+----------+--------------+-------------+---------------------+-------------------+---------------+-------------+
| CUS_0x1048| 27|   Accountant|     42387.54|  Standard|    0.04330683|   0.06137643|                  0.7|                 17|              1|   2024-02-01|
| CUS_0x10c0| 39|    Architect|     49454.13|       Bad|   0.028077738|  0.016946245|                 0.15|                 15|              1|   2024-02-01|
| CUS_0x115c| 20|     Engineer|     41695.76|       Bad|    0.10680965|   0.08901739|            1.0588236|                 22|              1|   2024-02-01|
| CUS_0x12b6| 18|    

In [22]:
feat_df.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- Annual_Income: float (nullable = true)
 |-- Monthly_Inhand_Salary: float (nullable = true)
 |-- Num_Bank_Accounts: integer (nullable = true)
 |-- Num_Credit_Card: integer (nullable = true)
 |-- Interest_Rate: integer (nullable = true)
 |-- Num_of_Loan: integer (nullable = true)
 |-- Delay_from_due_date: integer (nullable = true)
 |-- Num_of_Delayed_Payment: integer (nullable = true)
 |-- Changed_Credit_Limit: float (nullable = true)
 |-- Num_Credit_Inquiries: integer (nullable = true)
 |-- Credit_Mix: string (nullable = true)
 |-- Outstanding_Debt: float (nullable = true)
 |-- Credit_Utilization_Ratio: float (nullable = true)
 |-- Payment_of_Min_Amount: string (nullable = true)
 |-- Total_EMI_per_month: float (nullable = true)
 |-- Amount_invested_monthly: float (nullable = true)
 |-- Payment_Behaviour: string (nul

## sanity check: feature store + label store can train a model
join on Customer_ID (label snapshot_date is the mob 6 date, feature snapshot_date is the application date, each customer has one loan). train on 2023 applications, test on 2024 to make sure there is no leakage

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

lab_df = spark.read.parquet(*glob.glob(gold_label_store_directory + '*'))
joined = feat_df.join(lab_df.select("Customer_ID", "label"), on="Customer_ID", how="inner")
pdf = joined.toPandas()
print("joined rows:", len(pdf))

pdf["snapshot_date"] = pd.to_datetime(pdf.snapshot_date)
train = pdf[pdf.snapshot_date < "2024-01-01"]
test = pdf[pdf.snapshot_date >= "2024-01-01"]

drop_cols = ["Customer_ID", "snapshot_date", "label", "Occupation", "Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]
X_train = train.drop(columns=drop_cols).fillna(-999)
X_test = test.drop(columns=drop_cols).fillna(-999)

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, train.label)
auc = roc_auc_score(test.label, model.predict_proba(X_test)[:, 1])
print("train n:", len(train), "| test n:", len(test))
print("out of time test AUC:", round(auc, 4))

joined rows: 8974
train n: 5958 | test n: 3016
out of time test AUC: 0.8867


In [24]:
# which features matter most
imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
imp.head(10)

Outstanding_Debt        0.057844
Interest_Rate           0.041207
fe_5_mean               0.037190
fe_10_mean              0.028687
fe_9_mean               0.026526
num_credit_products     0.025216
debt_to_income          0.024230
Delay_from_due_date     0.023510
fe_4_mean               0.022623
Num_Credit_Inquiries    0.020482
dtype: float64